<a href="https://colab.research.google.com/github/alexandrumoldovan1/housing-prices-ml/blob/main/notebooks/04_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Install ML libraries
!pip install xgboost lightgbm catboost optuna -q

# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import joblib
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

# Boosting libraries
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

Libraries imported successfully!


In [14]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define paths
DRIVE_PATH = '/content/drive/MyDrive/ColabProjects/housing-prices-ml'
PROCESSED_DATA_PATH = f'{DRIVE_PATH}/processed_data'
MODELS_PATH = f'{DRIVE_PATH}/models'
OUTPUTS_PATH = f'{DRIVE_PATH}/outputs'

# Load processed datasets from Notebook 03
X_train = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_train.parquet')
X_test = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_test.parquet')
X_train_scaled = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_train_scaled.parquet')
X_test_scaled = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_test_scaled.parquet')
y_train = pd.read_parquet(f'{PROCESSED_DATA_PATH}/y_train.parquet').squeeze()
y_test = pd.read_parquet(f'{PROCESSED_DATA_PATH}/y_test.parquet').squeeze()

print(f"Data loaded successfully!\n")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data loaded successfully!

X_train: (99880, 30)
X_test:  (53522, 30)
y_train: (99880,)
y_test:  (53522,)


In [15]:
# Complete fix for data leakage
print("Fixing all data leakage sources...\n")

# Reload original data
df_raw = pd.read_parquet(f'{PROCESSED_DATA_PATH}/raw_data.parquet')
df_raw = df_raw[df_raw['SALE PRICE'] > 10000].copy()
lower = df_raw['SALE PRICE'].quantile(0.001)
upper = df_raw['SALE PRICE'].quantile(0.999)
df_raw = df_raw[(df_raw['SALE PRICE'] >= lower) & (df_raw['SALE PRICE'] <= upper)].copy()
df_raw = df_raw.reset_index(drop=True)

# Train/test masks
train_mask = df_raw['sale_year'].isin([2023, 2024])
test_mask = df_raw['sale_year'] == 2025

# FIX 1: Drop price_per_sqft completely (it's SALE_PRICE / sqft = direct leakage)
if 'price_per_sqft' in X_train.columns:
    X_train = X_train.drop(columns=['price_per_sqft'])
    X_test = X_test.drop(columns=['price_per_sqft'])
    print("Dropped price_per_sqft (direct leakage)")

# FIX 2: Recompute target encoding using ONLY train data
categorical_cols = ['NEIGHBORHOOD', 'BUILDING CLASS CATEGORY', 'TAX CLASS AT PRESENT',
                    'BUILDING CLASS AT PRESENT', 'BUILDING CLASS AT TIME OF SALE']

train_data = df_raw[train_mask].copy()
global_mean = train_data['SALE PRICE'].mean()

for col in categorical_cols:
    # Compute mean target per category, ONLY on train
    category_means = train_data.groupby(col)['SALE PRICE'].mean()
    # Map to train and test
    X_train[col] = df_raw.loc[train_mask, col].map(category_means).fillna(global_mean).values
    X_test[col] = df_raw.loc[test_mask, col].map(category_means).fillna(global_mean).values
    print(f"   Re-encoded {col} (train-only fit)")

# FIX 3: Recompute borough/neighborhood avg using only train (already done but let's redo to be sure)
borough_avg = train_data.groupby('borough_name')['SALE PRICE'].median()
neighborhood_avg = train_data.groupby('NEIGHBORHOOD')['SALE PRICE'].median()

X_train['borough_avg_price'] = df_raw.loc[train_mask, 'borough_name'].map(borough_avg).fillna(global_mean).values
X_test['borough_avg_price'] = df_raw.loc[test_mask, 'borough_name'].map(borough_avg).fillna(global_mean).values

X_train['neighborhood_avg_price'] = df_raw.loc[train_mask, 'NEIGHBORHOOD'].map(neighborhood_avg).fillna(global_mean).values
X_test['neighborhood_avg_price'] = df_raw.loc[test_mask, 'NEIGHBORHOOD'].map(neighborhood_avg).fillna(global_mean).values

print(f"\nFinal feature count: {X_train.shape[1]}")

# Re-scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# Save corrected versions
X_train.to_parquet(f'{PROCESSED_DATA_PATH}/X_train.parquet', compression='snappy', index=False)
X_test.to_parquet(f'{PROCESSED_DATA_PATH}/X_test.parquet', compression='snappy', index=False)
X_train_scaled.to_parquet(f'{PROCESSED_DATA_PATH}/X_train_scaled.parquet', compression='snappy', index=False)
X_test_scaled.to_parquet(f'{PROCESSED_DATA_PATH}/X_test_scaled.parquet', compression='snappy', index=False)
joblib.dump(scaler, f'{PROCESSED_DATA_PATH}/scaler.pkl')

print("\nAll leakage sources fixed!")
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")

Fixing all data leakage sources...

Dropped price_per_sqft (direct leakage)
   Re-encoded NEIGHBORHOOD (train-only fit)
   Re-encoded BUILDING CLASS CATEGORY (train-only fit)
   Re-encoded TAX CLASS AT PRESENT (train-only fit)
   Re-encoded BUILDING CLASS AT PRESENT (train-only fit)
   Re-encoded BUILDING CLASS AT TIME OF SALE (train-only fit)

Final feature count: 29

All leakage sources fixed!
   X_train: (99880, 29)
   X_test: (53522, 29)


In [16]:
# Reimport everything needed (in case session restarted)
import time
import joblib
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Define evaluation function (in case it was lost from session)
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name, use_cv=True, cv_folds=5):
    print(f"\n{'='*60}")
    print(f"Loading: {model_name}")
    print(f"{'='*60}")

    start_time = time.time()
    y_pred_log = model.predict(X_te)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_te)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_te, y_pred_log)

    cv_score = None
    if use_cv:
        kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
        cv_scores = cross_val_score(model, X_tr, y_tr, cv=kf, scoring='r2', n_jobs=-1)
        cv_score = cv_scores.mean()

    elapsed = time.time() - start_time
    print(f"   RMSE: ${rmse:,.0f}")
    print(f"   MAE:  ${mae:,.0f}")
    print(f"   R²:   {r2:.4f}")
    if cv_score is not None:
        print(f"   R² CV-{cv_folds}: {cv_score:.4f}")

    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R2_test': r2,
            'R2_cv': cv_score, 'Time_sec': elapsed, 'trained_model': model}

# Load already-trained models from Drive
print("Loading previously trained models from Drive...\n")
all_results = []

models_info = [
    ('linear_regression.pkl', 'Linear Regression', X_train_scaled),
    ('ridge.pkl', 'Ridge Regression', X_train_scaled),
    ('lasso.pkl', 'Lasso Regression', X_train_scaled),
    ('knn.pkl', 'KNN Regressor', X_train_scaled),
]

for filename, name, X_tr in models_info:
    model = joblib.load(f'{MODELS_PATH}/{filename}')
    result = evaluate_model(model, X_tr, y_train, X_test_scaled, y_test, name, use_cv=False)
    all_results.append(result)

print(f"\n{len(all_results)} models reloaded successfully.")

Loading previously trained models from Drive...


Loading: Linear Regression


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- price_per_sqft


In [17]:
# Initialize results list (we'll retrain all models from scratch with clean data)
all_results = []
print("Ready to train models with cleaned data (no leakage).")

Ready to train models with cleaned data (no leakage).


In [18]:
# Define a reusable evaluation function for all models
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name, use_cv=True, cv_folds=5):
    """
    Trains a model, evaluates on test set, and runs K-Fold Cross-Validation.
    Returns a dictionary with all metrics.
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    start_time = time.time()

    # Train on full training set
    model.fit(X_tr, y_tr)

    # Predict on test set (in log scale)
    y_pred_log = model.predict(X_te)

    # Convert predictions back to original $ scale
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_te)

    # Test set metrics (on original $ scale)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_te, y_pred_log)  # R² on log scale (more stable)

    # K-Fold Cross-Validation on training set (R² scoring)
    cv_score = None
    if use_cv:
        kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
        cv_scores = cross_val_score(model, X_tr, y_tr, cv=kf,
                                     scoring='r2', n_jobs=-1)
        cv_score = cv_scores.mean()

    elapsed = time.time() - start_time

    # Print results
    print(f"   RMSE (test):  ${rmse:,.0f}")
    print(f"   MAE  (test):  ${mae:,.0f}")
    print(f"   R²   (test):  {r2:.4f}")
    if cv_score is not None:
        print(f"   R²   (CV-{cv_folds}): {cv_score:.4f}")
    print(f"   Training time: {elapsed:.1f}s")

    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R2_test': r2,
        'R2_cv': cv_score,
        'Time_sec': elapsed,
        'trained_model': model
    }

# Initialize results storage
all_results = []

print("Evaluation function defined.")
print("\nMetrics explanation:")
print("   RMSE: Root Mean Squared Error (in $)")
print("   MAE:  Mean Absolute Error (in $)")
print("   R²:   Coefficient of determination (0-1, higher=better)")
print("   CV:   K-Fold Cross-Validation score")

Evaluation function defined.

Metrics explanation:
   RMSE: Root Mean Squared Error (in $)
   MAE:  Mean Absolute Error (in $)
   R²:   Coefficient of determination (0-1, higher=better)
   CV:   K-Fold Cross-Validation score


In [19]:
# Model 1: Linear Regression (baseline)
model = LinearRegression()
result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test,
                         "Linear Regression")
all_results.append(result)

# Save the trained model
joblib.dump(result['trained_model'], f'{MODELS_PATH}/linear_regression.pkl')
print(f"\nModel saved to: {MODELS_PATH}/linear_regression.pkl")


Training: Linear Regression
   RMSE (test):  $423,597,481
   MAE  (test):  $6,933,748
   R²   (test):  0.3530
   R²   (CV-5): 0.3437
   Training time: 5.5s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/linear_regression.pkl


In [20]:
# Model 2: Ridge Regression
def objective_ridge(trial):
    alpha = trial.suggest_float('alpha', 0.01, 100, log=True)
    model = Ridge(alpha=alpha, random_state=42)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning Ridge hyperparameters with Optuna (20 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_ridge, n_trials=20, show_progress_bar=False)
print(f"Best alpha: {study.best_params['alpha']:.4f}")

model = Ridge(alpha=study.best_params['alpha'], random_state=42)
result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test,
                         "Ridge Regression")
all_results.append(result)

joblib.dump(result['trained_model'], f'{MODELS_PATH}/ridge.pkl')
print(f"\nModel saved to: {MODELS_PATH}/ridge.pkl")

Tuning Ridge hyperparameters with Optuna (20 trials)...
Best alpha: 15.2815

Training: Ridge Regression
   RMSE (test):  $424,807,195
   MAE  (test):  $6,945,311
   R²   (test):  0.3530
   R²   (CV-5): 0.3437
   Training time: 0.4s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/ridge.pkl


In [21]:
# Model 3: Lasso Regression
def objective_lasso(trial):
    alpha = trial.suggest_float('alpha', 0.0001, 10, log=True)
    model = Lasso(alpha=alpha, random_state=42, max_iter=5000)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning Lasso hyperparameters with Optuna (20 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_lasso, n_trials=20, show_progress_bar=False)
print(f"Best alpha: {study.best_params['alpha']:.6f}")

model = Lasso(alpha=study.best_params['alpha'], random_state=42, max_iter=5000)
result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test,
                         "Lasso Regression")
all_results.append(result)

non_zero_features = np.sum(result['trained_model'].coef_ != 0)
total_features = len(result['trained_model'].coef_)
print(f"\nLasso selected {non_zero_features}/{total_features} features")

joblib.dump(result['trained_model'], f'{MODELS_PATH}/lasso.pkl')
print(f"Model saved to: {MODELS_PATH}/lasso.pkl")

Tuning Lasso hyperparameters with Optuna (20 trials)...
Best alpha: 0.000164

Training: Lasso Regression
   RMSE (test):  $424,659,324
   MAE  (test):  $6,942,698
   R²   (test):  0.3530
   R²   (CV-5): 0.3437
   Training time: 87.3s

Lasso selected 25/29 features
Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/lasso.pkl


In [22]:
# Model 4: KNN Regressor
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 5, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    p = trial.suggest_int('p', 1, 2)

    model = KNeighborsRegressor(n_neighbors=n_neighbors, weights=weights,
                                  p=p, n_jobs=-1)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning KNN hyperparameters with Optuna (15 trials)...")
print("This will take 10-20 minutes...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_knn, n_trials=15, show_progress_bar=False)
print(f"Best params: {study.best_params}")

model = KNeighborsRegressor(**study.best_params, n_jobs=-1)
result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test,
                         "KNN Regressor")
all_results.append(result)

joblib.dump(result['trained_model'], f'{MODELS_PATH}/knn.pkl')
print(f"Model saved to: {MODELS_PATH}/knn.pkl")

Tuning KNN hyperparameters with Optuna (15 trials)...
This will take 10-20 minutes...
Best params: {'n_neighbors': 17, 'weights': 'distance', 'p': 1}

Training: KNN Regressor
   RMSE (test):  $13,483,067
   MAE  (test):  $2,183,666
   R²   (test):  0.3770
   R²   (CV-5): 0.5918
   Training time: 474.0s
Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/knn.pkl


In [23]:
# Model 5: SVR (optimized for speed)
print("Preparing sample for SVR...")
sample_size = 15000
np.random.seed(42)
sample_idx = np.random.choice(X_train_scaled.index, size=sample_size, replace=False)
X_train_svr = X_train_scaled.loc[sample_idx]
y_train_svr = y_train.loc[sample_idx]

print(f"   Sample size: {len(X_train_svr):,} rows")

def objective_svr(trial):
    C = trial.suggest_float('C', 0.1, 10, log=True)
    epsilon = trial.suggest_float('epsilon', 0.01, 1.0, log=True)
    model = SVR(kernel='rbf', C=C, epsilon=epsilon, gamma='scale')
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_svr, y_train_svr, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("\nTuning SVR hyperparameters with Optuna (6 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_svr, n_trials=6, show_progress_bar=False)
print(f"Best params: {study.best_params}")

model = SVR(kernel='rbf', **study.best_params, gamma='scale')
print("\nTraining final SVR model...")
start_time = time.time()
model.fit(X_train_svr, y_train_svr)
print(f"   Trained in {time.time() - start_time:.1f}s")

result = evaluate_model(model, X_train_svr, y_train_svr, X_test_scaled, y_test,
                         "SVR (sample 15K)", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/svr.pkl')
print(f"\nModel saved to: {MODELS_PATH}/svr.pkl")

Preparing sample for SVR...
   Sample size: 15,000 rows

Tuning SVR hyperparameters with Optuna (6 trials)...
Best params: {'C': 8.01435621707864, 'epsilon': 0.12526368740530647}

Training final SVR model...
   Trained in 29.4s

Training: SVR (sample 15K)
   RMSE (test):  $13,795,433
   MAE  (test):  $2,300,141
   R²   (test):  0.2664
   Training time: 84.1s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/svr.pkl


In [24]:
# Model 6: Random Forest Regressor
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    max_depth = trial.suggest_int('max_depth', 10, 25)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    model = RandomForestRegressor(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
        n_jobs=-1, random_state=42
    )
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning Random Forest hyperparameters with Optuna (7 trials)...")
print("Estimated time: 30-50 minutes...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_rf, n_trials=7, show_progress_bar=False)
print(f"Best params: {study.best_params}")

print("\nTraining final Random Forest model...")
model = RandomForestRegressor(**study.best_params, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

result = evaluate_model(model, X_train, y_train, X_test, y_test,
                         "Random Forest", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/random_forest.pkl')
print(f"\nModel saved to: {MODELS_PATH}/random_forest.pkl")

Tuning Random Forest hyperparameters with Optuna (7 trials)...
Estimated time: 30-50 minutes...
Best params: {'n_estimators': 170, 'max_depth': 22, 'min_samples_split': 2, 'min_samples_leaf': 1}

Training final Random Forest model...

Training: Random Forest
   RMSE (test):  $13,636,641
   MAE  (test):  $2,054,461
   R²   (test):  0.4057
   Training time: 207.3s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/random_forest.pkl


In [25]:
# Model 7: XGBoost
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = XGBRegressor(**params, n_jobs=-1, random_state=42, verbosity=0)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning XGBoost hyperparameters with Optuna (10 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_xgb, n_trials=10, show_progress_bar=False)
print(f"Best params: {study.best_params}")

print("\nTraining final XGBoost model...")
model = XGBRegressor(**study.best_params, n_jobs=-1, random_state=42, verbosity=0)
result = evaluate_model(model, X_train, y_train, X_test, y_test,
                         "XGBoost", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/xgboost.pkl')
print(f"\nModel saved to: {MODELS_PATH}/xgboost.pkl")

Tuning XGBoost hyperparameters with Optuna (10 trials)...
Best params: {'n_estimators': 373, 'max_depth': 12, 'learning_rate': 0.01521417661701604, 'subsample': 0.7845932308155409, 'colsample_bytree': 0.6128966966838127, 'min_child_weight': 6}

Training final XGBoost model...

Training: XGBoost
   RMSE (test):  $13,638,059
   MAE  (test):  $2,048,680
   R²   (test):  0.4398
   Training time: 33.0s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/xgboost.pkl


In [26]:
# Model 8: LightGBM
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    model = LGBMRegressor(**params, n_jobs=-1, random_state=42, verbose=-1)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning LightGBM hyperparameters with Optuna (10 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_lgbm, n_trials=10, show_progress_bar=False)
print(f"Best params: {study.best_params}")

print("\nTraining final LightGBM model...")
model = LGBMRegressor(**study.best_params, n_jobs=-1, random_state=42, verbose=-1)
result = evaluate_model(model, X_train, y_train, X_test, y_test,
                         "LightGBM", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/lightgbm.pkl')
print(f"\nModel saved to: {MODELS_PATH}/lightgbm.pkl")

Tuning LightGBM hyperparameters with Optuna (10 trials)...
Best params: {'n_estimators': 394, 'max_depth': 12, 'learning_rate': 0.13326585940682364, 'num_leaves': 88, 'subsample': 0.6693567079735726, 'colsample_bytree': 0.8393401365793102}

Training final LightGBM model...

Training: LightGBM
   RMSE (test):  $13,617,110
   MAE  (test):  $2,054,535
   R²   (test):  0.4292
   Training time: 6.9s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/lightgbm.pkl


In [27]:
# Model 9: CatBoost
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 500),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
    }
    model = CatBoostRegressor(**params, random_state=42, verbose=0,
                                 thread_count=-1, bootstrap_type='Bernoulli')
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning CatBoost hyperparameters with Optuna (8 trials)...")
print("This may take 10-15 minutes...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_cat, n_trials=8, show_progress_bar=False)
print(f"Best params: {study.best_params}")

print("\nTraining final CatBoost model...")
model = CatBoostRegressor(**study.best_params, random_state=42, verbose=0,
                            thread_count=-1, bootstrap_type='Bernoulli')
result = evaluate_model(model, X_train, y_train, X_test, y_test,
                         "CatBoost", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/catboost.pkl')
print(f"\nModel saved to: {MODELS_PATH}/catboost.pkl")

Tuning CatBoost hyperparameters with Optuna (8 trials)...
This may take 10-15 minutes...
Best params: {'iterations': 211, 'depth': 8, 'learning_rate': 0.1777416196184352, 'l2_leaf_reg': 8.189544918587588, 'subsample': 0.8273584401862843}

Training final CatBoost model...

Training: CatBoost
   RMSE (test):  $13,623,366
   MAE  (test):  $2,090,907
   R²   (test):  0.4345
   Training time: 7.6s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/catboost.pkl


In [29]:
# Model 10: Stacking Regressor (combines top 3 boosting models)
print("Building Stacking Regressor...")
print("Base learners: XGBoost, LightGBM, CatBoost")
print("Meta-learner: Ridge Regression\n")

# Load the best trained models directly (no need to recreate)
xgb_model = joblib.load(f'{MODELS_PATH}/xgboost.pkl')
lgbm_model = joblib.load(f'{MODELS_PATH}/lightgbm.pkl')
cat_model = joblib.load(f'{MODELS_PATH}/catboost.pkl')

base_models = [
    ('xgb', xgb_model),
    ('lgbm', lgbm_model),
    ('cat', cat_model)
]

meta_model = Ridge(alpha=1.0)

stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=3,
    n_jobs=-1
)

print("Training Stacking Regressor (this may take 10-20 minutes)...")
result = evaluate_model(stacking_model, X_train, y_train, X_test, y_test,
                         "Stacking Regressor", use_cv=False)
all_results.append(result)

joblib.dump(result['trained_model'], f'{MODELS_PATH}/stacking.pkl')
print(f"\nModel saved to: {MODELS_PATH}/stacking.pkl")

Building Stacking Regressor...
Base learners: XGBoost, LightGBM, CatBoost
Meta-learner: Ridge Regression

Training Stacking Regressor (this may take 10-20 minutes)...

Training: Stacking Regressor
   RMSE (test):  $13,621,226
   MAE  (test):  $2,040,665
   R²   (test):  0.4485
   Training time: 151.7s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/stacking.pkl


In [30]:
# Create final comparison table of all 10 models
print("="*70)
print("NOTEBOOK 04 - MODEL TRAINING SUMMARY")
print("="*70)

# Build results DataFrame
results_df = pd.DataFrame(all_results)
results_df = results_df[['Model', 'R2_test', 'R2_cv', 'RMSE', 'MAE', 'Time_sec']]
results_df.columns = ['Model', 'R² (test)', 'R² (CV-5)', 'RMSE ($)', 'MAE ($)', 'Time (s)']
results_df = results_df.sort_values('R² (test)', ascending=False).reset_index(drop=True)

# Format for display
display_df = results_df.copy()
display_df['R² (test)'] = display_df['R² (test)'].round(4)
display_df['R² (CV-5)'] = display_df['R² (CV-5)'].apply(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
display_df['RMSE ($)'] = display_df['RMSE ($)'].apply(lambda x: f'${x:,.0f}')
display_df['MAE ($)'] = display_df['MAE ($)'].apply(lambda x: f'${x:,.0f}')
display_df['Time (s)'] = display_df['Time (s)'].round(1)

print(f"\nAll 10 models ranked by R² (test):\n")
print(display_df.to_string(index=False))

# Save results to CSV for Notebook 05
results_df.drop(columns=['Time (s)'] if 'Time (s)' in results_df.columns else []).to_csv(
    f'{OUTPUTS_PATH}/model_results.csv', index=False
)

# Save full results with model objects removed (keep only metrics)
results_for_save = []
for r in all_results:
    r_copy = {k: v for k, v in r.items() if k != 'trained_model'}
    results_for_save.append(r_copy)

pd.DataFrame(results_for_save).to_csv(f'{OUTPUTS_PATH}/model_results.csv', index=False)

print(f"\n\nResults saved to: {OUTPUTS_PATH}/model_results.csv")
print(f"All 10 models saved to: {MODELS_PATH}/")

print("\n" + "="*70)
print("Notebook 04 completed successfully.")
print("Next step: Notebook 05 - Model Evaluation & Interpretation (SHAP, feature importance)")
print("="*70)

NOTEBOOK 04 - MODEL TRAINING SUMMARY

All 10 models ranked by R² (test):

             Model  R² (test) R² (CV-5)     RMSE ($)    MAE ($)  Time (s)
Stacking Regressor     0.4485       N/A  $13,621,226 $2,040,665     151.7
           XGBoost     0.4398       N/A  $13,638,059 $2,048,680      33.0
          CatBoost     0.4345       N/A  $13,623,366 $2,090,907       7.6
          LightGBM     0.4292       N/A  $13,617,110 $2,054,535       6.9
     Random Forest     0.4057       N/A  $13,636,641 $2,054,461     207.3
     KNN Regressor     0.3770    0.5918  $13,483,067 $2,183,666     474.0
  Ridge Regression     0.3530    0.3437 $424,807,195 $6,945,311       0.4
  Lasso Regression     0.3530    0.3437 $424,659,324 $6,942,698      87.3
 Linear Regression     0.3530    0.3437 $423,597,481 $6,933,748       5.5
  SVR (sample 15K)     0.2664       N/A  $13,795,433 $2,300,141      84.1


Results saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/outputs/model_results.csv
All 10 mode